# 40. Users with three consecutive login days

**Hard**

Return each user_id once when they have any streak of at least three distinct consecutive calendar dates. Repeated logins on one date do not extend a streak. Handle month/year boundaries and multiple qualifying streaks.


In [0]:
%run ./_setup $exercise_id="40"


**Your solution:** add imports and return your result from `solve`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def solve(input_df):
    # Write your solution here.
    return (
        input_df
        .withColumn("login_date1",F.dayofyear("login_date"))
        .withColumn("last_login",F.col("login_date1") - F.lag("login_date1").over(Window.partitionBy("user_id").orderBy("login_date1")))
        .withColumn("next_login",F.lead("login_date1").over(Window.partitionBy("user_id").orderBy("login_date1")) - F.col("login_date1")   )
        .filter(
            (F.col("last_login")==1) &
            (F.col("next_login")==1)
        )
        .select("user_id").distinct()
    )

solve(input_df).display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w=Window.partitionBy("user_id").orderBy("date")

def solve(input_df):
    # Write your solution here.
    return (
        input_df
        .select("user_id",F.to_date("login_date").alias("date")).distinct()
        .withColumn("next_to_next_login", F.lead("date",2).over(w))
        .filter(
            F.date_add("date",2) == F.col("next_to_next_login")
        )
        .select("user_id").distinct()
    )

solve(input_df).display()

### ## # ## using classic gaps and islands approach

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w=Window.partitionBy("user_id").orderBy("date")

def solve(input_df):
    # Write your solution here.
    return (
        input_df
        .select("user_id",F.to_date("login_date").alias("date")).distinct()
        .withColumn("islands",F.date_sub(F.col("date"), F.row_number().over(w)))
        .groupBy("user_id","islands")
        .agg(F.count("*").alias("streak_length"))
        .filter(F.col("streak_length")>=3)
        .select("user_id").distinct()
    )

solve(input_df).display()

**Check:** run below to test all cases and save progress.


In [0]:
check_report = check_solution(exercise_id, solve, spark)


In [0]:
show_progress(exercise_id, spark)
